<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/ShortTrades.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [69]:
!pip install --upgrade yfinance
!pip install  --upgrade pandas_ta
!pip install pandas-ta
!pip install ta
!pip install scipy==1.16.2

In [70]:
import yfinance as yf
print(yf.__version__)
import pandas as pd
import ta
import numpy as np
import requests
from datetime import datetime, timedelta
from scipy.stats import linregress
from transformers import pipeline
import time
import random
random.seed(42)
print("Libraries Installed!")

1.3.0
Libraries Installed!


In [71]:
today = datetime.today()
start_of_year = today.replace(month=1, day=1)

# First day of this month
first_day_month = today.replace(day=1)

# First day of this week (Monday as weekday 0)
first_day_week = today - timedelta(days=today.weekday())
print("First day of year:", start_of_year)

print("\nFirst day of this month:", first_day_month)
print("\nFirst day of this week:", first_day_week)


def most_recent_quarter_start(today=None):
    if today is None:
        today = pd.Timestamp.today().normalize()
    year = today.year
    month = today.month

    # Determine quarter start months: Jan, Apr, Jul, Oct
    if month >= 10:
        q_start = pd.Timestamp(year, 10, 1)
    elif month >= 7:
        q_start = pd.Timestamp(year, 7, 1)
    elif month >= 4:
        q_start = pd.Timestamp(year, 4, 1)
    else:
        q_start = pd.Timestamp(year, 1, 1)

    return q_start

# Example usage
print("Today:", pd.Timestamp.today().normalize())
print("Most recent quarter start:", most_recent_quarter_start())

First day of year: 2026-01-01 00:34:04.272115

First day of this month: 2026-05-01 00:34:04.272115

First day of this week: 2026-05-04 00:34:04.272115
Today: 2026-05-10 00:00:00
Most recent quarter start: 2026-04-01 00:00:00


In [72]:
# List of ETFs to analyze
recent_quarter = most_recent_quarter_start()
start_of_year = '2025-01-01'
df_raw = pd.read_csv('short_list.csv')
df_raw = df_raw[df_raw['Type'].isin(['ETF','Stock','ASX','TSX','AS'])]
#df_raw = df_raw[df_raw['Type'].isin(['TSX'])]
df_raw = df_raw.drop_duplicates(subset=['Asset'])
etfs = df_raw['Asset'].to_list()
print(etfs)

print(len(etfs))

['ASM.AS', 'AKZA.AS', 'EXO.AS', 'HEIA.AS', 'PRX.AS', 'PHIA.AS', 'A2M.AX', 'SNZ.AX', 'TPW.AX', 'SHL.AX', 'BRG.AX', 'SUL.AX', 'RMD.AX', 'LNW.AX', 'HVN.AX', 'REG.AX', 'LOV.AX', 'JBH.AX', 'EDV.AX', 'PMV.AX', 'FPH.AX', 'TLC.AX', 'EVT.AX', 'NEU.AX', 'WES.AX', 'TWE.AX', 'GIL.TO', 'COR', 'PODD', 'TECH', 'MCK', 'SYK', 'ABT', 'BSX', 'MDT', 'COO', 'BAX', 'ZTS', 'DHR', 'BMY', 'BDX', 'CAH', 'HCA', 'ABBV', 'HSIC', 'STE', 'DGX', 'DXCM', 'ISRG', 'LH', 'TMO', 'AMGN', 'JNJ', 'CI', 'ALGN', 'EW', 'PFE', 'EXE', 'TPL', 'OXY', 'EQT', 'KMI', 'COP', 'WMB', 'CVX', 'XOM', 'FE', 'DTE', 'NRG', 'PCG', 'ES', 'D', 'ED', 'AWK', 'PEG', 'CMS', 'EXC', 'AEE', 'SO', 'DUK', 'WEC', 'XEL', 'PNW', 'CNP', 'AEP', 'PPL', 'EVRG', 'SRE', 'ATO', 'VST', 'EIX', 'VMC', 'MKC', 'CAG', 'HRL', 'CLX', 'GIS', 'SYY', 'DLTR', 'CPB', 'SJM', 'DG', 'CHD', 'HSY', 'KDP', 'LW', 'KR', 'PEP', 'STZ', 'MNST', 'COST', 'CL', 'KVUE', 'KO', 'FISV', 'AJG', 'BRO', 'JKHY', 'FDS', 'ERIE', 'WTW', 'AON', 'L', 'GPN', 'PGR', 'ICE', 'FIS', 'MCO', 'HIG', 'MA', 'WRB',

In [73]:

def weinstein_stage(df, sma_window=30):
    """Determine Weinstein stage using 30-week SMA and its slope."""
     # Handle MultiIndex columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df["SMA"] = df["Close"].rolling(window=sma_window).mean()

    # Compute linear regression slope on last N SMA points
    if len(df.dropna()) < sma_window:
        return None  # not enough data

    slope, _, _, _, _ = linregress(range(sma_window-25), df["SMA"].tail(sma_window-25))

    latest_price = df["Close"].iloc[-1]
    latest_sma = df["SMA"].iloc[-1]

    # Determine stage
    if latest_price > latest_sma and slope > 0:
        stage = "Stage 2 (Advancing)"
    elif latest_price < latest_sma and slope < 0:
        stage = "Stage 4 (Declining)"
    elif  np.abs(slope) <= 0.001:
        stage = "Stage 1 (Basing)"
    else:
        stage = "Stage 3 (Topping)"

    return stage, slope, latest_price, latest_sma


In [74]:
# Classify stocks into stages
results = []
for etf in etfs:
    df = yf.download(etf, period="3y", interval="1wk", auto_adjust=True)
    stage_info = weinstein_stage(df)
    if stage_info:
        stage, slope, price, sma = stage_info
        results.append({
            "ETF": etf,
            "Stage": stage,
            "SMA_Slope": slope,
            "Latest_Price": price,
            "30W_SMA": sma
        })

stages_df = pd.DataFrame(results).sort_values(by="SMA_Slope", ascending=False)
stages_df.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,ETF,Stage,SMA_Slope,Latest_Price,30W_SMA
0,ASM.AS,Stage 2 (Advancing),10.877333,882.200012,641.149994
58,TPL,Stage 2 (Advancing),3.946520,394.200012,381.667008
110,COST,Stage 2 (Advancing),2.856505,1008.789978,952.412543
30,MCK,Stage 3 (Topping),2.416136,736.090027,854.679523
158,TDY,Stage 2 (Advancing),2.071867,621.380005,584.903662


In [75]:
declining_stocks= stages_df[stages_df["Stage"] .isin(["Stage 4 (Declining)"]) ]
declining_stocks.reset_index(drop=True, inplace=True)
declining_stocks.head()

,ETF,Stage,SMA_Slope,Latest_Price,30W_SMA
0,EDV.AX,Stage 4 (Declining),-0.004831,3.250000,3.557113
1,PHIA.AS,Stage 4 (Declining),-0.005800,23.120001,24.217333
2,TLC.AX,Stage 4 (Declining),-0.005945,5.300000,5.312566
3,PEG,Stage 4 (Declining),-0.014512,77.129997,80.939918
4,REG.AX,Stage 4 (Declining),-0.014685,6.350000,6.746460


In [76]:
# List of ETFs to analyze
df_o = df_raw[df_raw['Asset'].isin(declining_stocks['ETF'])]
df_raw = df_o.copy()
etfs2 = df_raw['Asset'].to_list()
etfs = list(dict.fromkeys(etfs2))

print(etfs)

print(len(etfs))

['AKZA.AS', 'EXO.AS', 'PRX.AS', 'PHIA.AS', 'A2M.AX', 'SNZ.AX', 'TPW.AX', 'SHL.AX', 'BRG.AX', 'SUL.AX', 'RMD.AX', 'LNW.AX', 'HVN.AX', 'REG.AX', 'LOV.AX', 'JBH.AX', 'EDV.AX', 'PMV.AX', 'FPH.AX', 'TLC.AX', 'EVT.AX', 'NEU.AX', 'WES.AX', 'TWE.AX', 'COR', 'PODD', 'TECH', 'SYK', 'ABT', 'BSX', 'MDT', 'COO', 'BAX', 'ZTS', 'DHR', 'ABBV', 'STE', 'DXCM', 'LH', 'TMO', 'EXE', 'NRG', 'AWK', 'PEG', 'VST', 'VMC', 'MKC', 'CAG', 'HRL', 'CLX', 'GIS', 'SYY', 'CPB', 'SJM', 'HSY', 'LW', 'FISV', 'AJG', 'BRO', 'FDS', 'ERIE', 'WTW', 'AON', 'GPN', 'PGR', 'ICE', 'FIS', 'MCO', 'MA', 'WRB', 'SPGI', 'AIG', 'CDW', 'EPAM', 'TYL', 'GDDY', 'CTSH', 'TRMB', 'PLTR', 'WDAY', 'ACN', 'NOW', 'IBM', 'INTU', 'PTC', 'GEN', 'MSI', 'CRM', 'ADBE', 'IT']
90


In [77]:
def macdv(prices, fast=12, slow=26, signal=9, atr_window=10, thresholds=(50, 150)):
    """
    Compute MACD-V (volatility normalized MACD).

    Parameters
    ----------
    prices : pd.Series
        Price series (e.g. closing prices).
    fast : int
        Fast EMA period.
    slow : int
        Slow EMA period.
    signal : int
        Signal EMA period for MACD line.
    atr_window : int
        ATR lookback window.
    thresholds : tuple
        (lower, upper) thresholds for neutral/ranging and extreme momentum zones.

    Returns
    -------
    pd.DataFrame with columns:
        - MACDV : MACD-V value
        - Signal : EMA of MACDV
        - Histogram : MACDV - Signal
        - EntryFlag : True when momentum is strong enough, False otherwise
    """
    # --- Step 1: EMAs for MACD ---
    ema_fast = prices.ewm(span=fast, adjust=False).mean()
    ema_slow = prices.ewm(span=slow, adjust=False).mean()
    macd_raw = ema_fast - ema_slow

    # --- Step 2: ATR for normalization ---
    high = prices.shift(1) * (1 + 0.01)   # synthetic highs/lows if OHLC not available
    low = prices.shift(1) * (1 - 0.01)
    close = prices
    tr1 = high - low
    tr2 = (high - close.shift(1)).abs()
    tr3 = (low - close.shift(1)).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(atr_window).mean()

    # --- Step 3: Normalize MACD by ATR ---
    macdv = (macd_raw / atr) * 100

    # --- Step 4: Signal line and histogram ---
    signal_line = macdv.ewm(span=signal, adjust=False).mean()
    histogram = macdv - signal_line

    # --- Step 5: Entry conditions ---
    lower, upper = thresholds
    entry_flag = ((macdv.abs() > lower) & (macdv.abs() < upper)) | (macdv.abs() > upper)

    df = pd.DataFrame({
        "MACDV": macdv,
        "Signal": signal_line,
        "Histogram": histogram,
        "EntryFlag": entry_flag
    })
    curr = df.iloc[-1]

    return curr.EntryFlag

def money_flow_signals(df, period=10):
    """
    Calculate Money Flow Index (MFI) and generate signals:
    - Positive money flow (TP > TP_prev)
    - Divergence (Price vs MFI mismatch)

    Parameters:
        df (pd.DataFrame): DataFrame with columns ["High", "Low", "Close", "Volume"]
        period (int): Lookback period for MFI (default=10)

    Returns:
        pd.DataFrame with added columns: ["TypicalPrice", "MFI", "PositiveFlow", "Divergence"]
    """

    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level

    # Step 1: Typical Price
    df["TypicalPrice"] = (df["High"] + df["Low"] + df["Close"]) / 3

    # Step 2: Raw Money Flow
    df["RawMoneyFlow"] = df["TypicalPrice"] * df["Volume"]

    # Step 3: Positive & Negative Flow
    df["PositiveFlow"] = np.where(df["TypicalPrice"] > df["TypicalPrice"].shift(1), df["RawMoneyFlow"], 0.0)
    df["NegativeFlow"] = np.where(df["TypicalPrice"] < df["TypicalPrice"].shift(1), df["RawMoneyFlow"], 0.0)

    # Step 4: Money Flow Ratio & MFI
    pos_flow = df["PositiveFlow"].rolling(period).sum()
    neg_flow = df["NegativeFlow"].rolling(period).sum()
    money_flow_ratio = pos_flow / neg_flow.replace(0, np.nan)
    df["MFI"] = 100 - (100 / (1 + money_flow_ratio))

    # Step 5: Positive Money Flow Signal
    df["PositiveFlowSignal"] = df["TypicalPrice"] > df["TypicalPrice"].shift(1)

    # Step 6: Divergence Detection
    df["PriceHigh"] = df["Close"].rolling(period).max()
    df["PriceLow"] = df["Close"].rolling(period).min()
    df["MFIHigh"] = df["MFI"].rolling(period).max()
    df["MFILow"] = df["MFI"].rolling(period).min()

    def divergence(row):
        if np.isnan(row["MFI"]):
            return None
        # Price makes higher high, but MFI does not
        if row["Close"] >= row["PriceHigh"] and row["MFI"] < row["MFIHigh"]:
            return "Bearish Divergence ⚠️"
        # Price makes lower low, but MFI does not
        elif row["Close"] <= row["PriceLow"] and row["MFI"] > row["MFILow"]:
            return "Bullish Divergence ✅"
        else:
            return "No Divergence"

    df["Divergence"] = df.apply(divergence, axis=1)
    df['Entry_signal']= (df["MFI"] > 50) & (df["Divergence"] == "No Divergence")
    curr = df.iloc[-1]

    return curr.Entry_signal

def anchored_vwap_old(ticker, anchor_date):
    """
    Calculate Anchored VWAP starting from a given anchor_date.
    Works with both single-level and multi-level columns (e.g. yfinance output).
    """
    # --- Step 1: Flatten columns if multi-index ---
    df = yf.download(ticker, period="1y", interval="1d",auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level

    # --- Step 2: Ensure required columns exist ---
    required_cols = ["High", "Low", "Close", "Volume"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    # --- Step 3: Subset from anchor_date ---
    df_anchor = df.loc[df.index >= pd.to_datetime(anchor_date)].copy()
    if df_anchor.empty:
        raise ValueError(f"No data found on/after {anchor_date}")

    # --- Step 4: Compute typical price ---
    df_anchor["typical_price"] = (df_anchor["High"] + df_anchor["Low"] + df_anchor["Close"]) / 3.0

    # --- Step 5: Cumulative PV and VWAP ---
    df_anchor["cum_pv"] = (df_anchor["typical_price"].astype(float) * df_anchor["Volume"].astype(float)).cumsum()
    df_anchor["cum_vol"] = df_anchor["Volume"].astype(float).cumsum()
    df_anchor["anchored_vwap"] = df_anchor["cum_pv"] / df_anchor["cum_vol"]

    return df_anchor[["anchored_vwap"]]


In [78]:
# Filter ETFs or stocks for liquidity
def filter_by_liquidity(etf_df, ticker_col="Asset", min_dollar_vol=1e6, lookback_days=30):
    liquid_etfs = []

    for ticker in etf_df[ticker_col]:
        try:
            # Fetch daily historical data
            data = yf.download(ticker, period=f"{lookback_days*2}d", interval="1d", auto_adjust=True)

            if data.empty:
                continue

            # Calculate dollar volume (Close × Volume)
            data["dollar_volume"] = data["Close"] * data["Volume"]

            # Calculate rolling average over lookback_days
            avg_dollar_volume = data["dollar_volume"].rolling(window=lookback_days).mean().iloc[-1]

            # Check liquidity condition
            if avg_dollar_volume >= min_dollar_vol:
                liquid_etfs.append(ticker)

        except Exception as e:
            print(f"Error fetching {ticker}: {e}")

    # Return filtered DataFrame
    return etf_df[etf_df[ticker_col].isin(liquid_etfs)]

# Example usage
df = pd.DataFrame({"Assets": etfs})
liquid_df = filter_by_liquidity(df, ticker_col="Assets")
df_o = df_raw[df_raw['Asset'].isin(liquid_df['Assets'])]
etfs2 = df_o['Asset'].to_list()
etfs_clean = list(dict.fromkeys(etfs2))


print("")
print(etfs_clean)
print(len(etfs_clean))



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********


['AKZA.AS', 'EXO.AS', 'PRX.AS', 'PHIA.AS', 'A2M.AX', 'TPW.AX', 'SHL.AX', 'BRG.AX', 'SUL.AX', 'RMD.AX', 'LNW.AX', 'HVN.AX', 'REG.AX', 'LOV.AX', 'JBH.AX', 'EDV.AX', 'PMV.AX', 'FPH.AX', 'TLC.AX', 'EVT.AX', 'NEU.AX', 'WES.AX', 'TWE.AX', 'COR', 'PODD', 'TECH', 'SYK', 'ABT', 'BSX', 'MDT', 'COO', 'BAX', 'ZTS', 'DHR', 'ABBV', 'STE', 'DXCM', 'LH', 'TMO', 'EXE', 'NRG', 'AWK', 'PEG', 'VST', 'VMC', 'MKC', 'CAG', 'HRL', 'CLX', 'GIS', 'SYY', 'CPB', 'SJM', 'HSY', 'LW', 'FISV', 'AJG', 'BRO', 'FDS', 'ERIE', 'WTW', 'AON', 'GPN', 'PGR', 'ICE', 'FIS', 'MCO', 'MA', 'WRB', 'SPGI', 'AIG', 'CDW', 'EPAM', 'TYL', 'GDDY', 'CTSH', 'TRMB', 'PLTR', 'WDAY', 'ACN', 'NOW', 'IBM', 'INTU', 'PTC', 'GEN', 'MSI', 'CRM', 'ADBE', 'IT']
89


In [95]:

def anchored_vwap_structural(
    ticker: str,
    lookback_weeks: int = 5,
    pivot_left: int = 2,
    pivot_right: int = 2):
    """
    Anchors VWAP from the last STRUCTURAL swing low
    that led to a Lower High (LH), within a lookback window.
    """

    try:
        # ----------------------------
        # 1. Download data
        # ----------------------------
        data = yf.download(
            ticker,
            period="3mo",
            interval="1d",
            auto_adjust=True,
            progress=False
        )

        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)

        data.dropna(inplace=True)
        data.index = pd.to_datetime(data.index)

        lookback_days = lookback_weeks * 5
        if len(data) < lookback_days:
            raise ValueError("Not enough data")

        recent = data.tail(lookback_days)

        # ----------------------------
        # 2. Find STRUCTURAL swing lows
        # ----------------------------
        swing_lows = []
        for i in range(pivot_left, len(recent) - pivot_right):
            window = recent['Low'].iloc[i - pivot_left : i + pivot_right + 1]
            if recent['Low'].iloc[i] == window.min():
                swing_lows.append(recent.index[i])

        if not swing_lows:
            raise ValueError("No swing lows found")

        # ----------------------------
        # 3. Find LL that caused a LH
        # ----------------------------
        anchor_date = None

        for sl in reversed(swing_lows):
            after_sl = recent.loc[sl:]

            highs = after_sl['High']
            for i in range(1, len(highs)):
                # LH definition: failed attempt to make HH
                if highs.iloc[i] < highs.iloc[i - 1]:
                    anchor_date = sl
                    break

            if anchor_date is not None:
                break

        if anchor_date is None:
            # No structural breakdown
            data['Anchored_VWAP'] = np.nan
            data['Signal'] = False
            return data[['Anchored_VWAP', 'Signal']]

        # ----------------------------
        # 4. Anchor VWAP from STRUCTURAL LL
        # ----------------------------
        anchor_data = data.loc[anchor_date:]

        typical_price = (
            anchor_data['High']
            + anchor_data['Low']
            + anchor_data['Close']
        ) / 3

        volume = anchor_data['Volume']

        pv = (typical_price * volume).cumsum()
        v = volume.cumsum()

        avwap = pv / v.where(v != 0, np.nan)

        data['Anchored_VWAP'] = np.nan
        data.loc[anchor_date:, 'Anchored_VWAP'] = avwap

        # ----------------------------
        # 5. Final signal logic
        # ----------------------------
        latest_close = data['Close'].iloc[-1]
        swing_low_price = data.loc[anchor_date, 'Low']

        data['Signal'] = (
            (data['Close'] < data['Anchored_VWAP']) &
            (latest_close < swing_low_price) &
            (data['Anchored_VWAP'].notna())
        )

        print(
            f"{ticker} | AVWAP anchored from {anchor_date.date()} "
            f"(structural LL @ {swing_low_price:.2f})"
        )

        return data[['Anchored_VWAP', 'Signal']]

    except Exception as e:
        print(f"{ticker} error: {e}")
        return None


# Function to fetch historical weekly data
def rolling_regression_slope(series, window=10):
    """Rolling linear regression slope (price units per bar)."""
    def calc_slope(y):
        if len(y) < 2:
            return np.nan
        x = np.arange(len(y))
        slope, _, _, _, _ = linregress(x, y)
        return slope
    return series.rolling(window).apply(calc_slope, raw=False)


def anchored_vwap_old(ticker, anchor_date):
    """
    Calculate Anchored VWAP starting from a given anchor_date.
    Works with both single-level and multi-level columns (e.g. yfinance output).
    """
    # --- Step 1: Flatten columns if multi-index ---
    df = yf.download(ticker, period="1y", interval="1d",auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level

    # --- Step 2: Ensure required columns exist ---
    required_cols = ["High", "Low", "Close", "Volume"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    # --- Step 3: Subset from anchor_date ---
    df_anchor = df.loc[df.index >= pd.to_datetime(anchor_date)].copy()
    if df_anchor.empty:
        raise ValueError(f"No data found on/after {anchor_date}")

    # --- Step 4: Compute typical price ---
    df_anchor["typical_price"] = (df_anchor["High"] + df_anchor["Low"] + df_anchor["Close"]) / 3.0

    # --- Step 5: Cumulative PV and VWAP ---
    df_anchor["cum_pv"] = (df_anchor["typical_price"].astype(float) * df_anchor["Volume"].astype(float)).cumsum()
    df_anchor["cum_vol"] = df_anchor["Volume"].astype(float).cumsum()
    df_anchor["anchored_vwap"] = df_anchor["cum_pv"] / df_anchor["cum_vol"]

    return df_anchor[["anchored_vwap"]]

def anchored_vwap(ticker: str, lookback_weeks: int = 5):
    """
    Calculate the Anchored VWAP from the most recent high within the past `lookback_weeks`
    and create a 'Signal' column that gives 'Buy' when Close > Anchored_VWAP, else 'No-Buy'.

    Args:
        ticker (str): Stock ticker symbol
        lookback_weeks (int): Number of weeks to look back for the highest price

    Returns:
        pandas.DataFrame: DataFrame with OHLC, Volume, Anchored_VWAP, and Signal columns
    """
    try:
        # --- Fetch 6 months of daily data to cover the lookback window
        data = yf.download(ticker, period="3mo", interval="1d", progress=False,auto_adjust=True)
        if isinstance(data.columns, pd.MultiIndex):
          data.columns = data.columns.get_level_values(0)  # keep only first level

        if data.empty:
            raise ValueError(f"No data retrieved for ticker {ticker}")

        data.dropna(inplace=True)
        data.index = pd.to_datetime(data.index)

        # --- Ensure we have enough data for the lookback period
        min_days_required = lookback_weeks * 5  # ~5 trading days per week
        if len(data) < min_days_required:
            raise ValueError(f"Insufficient data: {len(data)} days available, {min_days_required} required")

        # --- Find the most recent high within the past lookback_weeks
        recent_period = data.tail(min_days_required)
        anchor_date = recent_period['Low'].idxmin()

        # --- Verify anchor_date is a valid Timestamp
        if not isinstance(anchor_date, pd.Timestamp):
            raise ValueError(f"Invalid anchor_date: {anchor_date}. Expected a Timestamp.")

        #anchor_price = recent_period.loc[anchor_date, 'Low']
        anchor_price = recent_period['Low'].min()

        # --- Use .date() safely since we confirmed anchor_date is a Timestamp
        print(f"Anchored VWAP for {ticker} starting from {anchor_date.date()} (recent low = {anchor_price:.2f})")

        # --- Slice data from the anchor date onwards
        anchor_data = data.loc[anchor_date:]

        # --- Compute VWAP starting from the anchor date
        # Use typical price ((H+L+C)/3) for more accurate VWAP
        typical_price = (anchor_data['High'] + anchor_data['Low'] + anchor_data['Close']) / 3
        q = anchor_data['Volume']
        pv = (typical_price * q).cumsum()
        v = q.cumsum()

        # --- Avoid division by zero
        avwap = pv / v.where(v != 0, np.nan)

        # --- Add Anchored VWAP to the full dataset
        data['Anchored_VWAP'] = np.nan  # Initialize with NaN
        data.loc[anchor_date:, 'Anchored_VWAP'] = avwap

        # --- Create Buy/No-Buy Signal
        # Only apply signal where Anchored_VWAP is not NaN
        data['Signal'] = np.where(
            (data['Close'] < data['Anchored_VWAP']) & (data['Anchored_VWAP'].notna()),
            True,
            False
        )

        return data[['Anchored_VWAP', 'Signal']]

    except Exception as e:
        print(f"Error processing {ticker}: {str(e)}")
        return None

def get_monthly_data(ticker):
  try:
      df = yf.download(ticker, period="10y", interval="1mo",auto_adjust=True)

      df['10_month_SMA'] = df['Close'].rolling(window=10).mean()
      # Calculate MACD and Signal Line
      df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
      df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
      # Compute MACD Line
      df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
      # Compute Signal Line (9-day EMA of MACD Line)
      df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
      df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']
      df['RVOL'] = df['Volume'] / df['Volume'].rolling(window=10).mean()
      df['RVOL_Slope'] = df['RVOL'].diff()
      # Calculate ADX, +DMI and -DMI
      high = df['High']
      low = df['Low']
      close = df['Close']
      # Calculate directional movements
      up_move = high.diff()
      down_move = -low.diff()
      plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
      minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
      # Calculate True Range (TR)
      tr1 = high - low
      tr2 = (high - close.shift()).abs()
      tr3 = (low - close.shift()).abs()
      tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
      # Smooth TR, +DM, and -DM using Wilder’s smoothing
      atr = tr.rolling(window=14).sum()
      plus_dm_series = pd.Series(plus_dm.ravel(), index=df.index).astype(float)
      minus_dm_series = pd.Series(minus_dm.ravel(), index=df.index).astype(float)
      plus_dm_smoothed = plus_dm_series.rolling(window=14).sum()
      minus_dm_smoothed = minus_dm_series.rolling(window=14).sum()
      # Directional Indicators
      plus_di = 100 * (plus_dm_smoothed / atr)
      minus_di = 100 * (minus_dm_smoothed / atr)
      # DX and ADX
      dx = 100 * (np.abs(plus_di - minus_di) / (plus_di + minus_di))
      adx = dx.rolling(window=14).mean()
      # Add results to original DataFrame
      df['+DI'] = plus_di
      df['-DI'] = minus_di
      df['ADX'] = adx
      df['di_flag'] = df['-DI'] > df['+DI']
      df['di_flag'] = df['di_flag'].astype(int)
      df['adx_indicator'] = np.where(df['ADX'] > 10, 1, 0)
      df['adx_signal'] = df['adx_indicator'] * df['di_flag']

      return df
  except Exception as e:
      print("There is an error getting monthly data", e)

def get_weekly_data(ticker):
  try:
      df = yf.download(ticker, period="2y", interval="1wk",auto_adjust=True)

      df['10_week_SMA'] = df['Close'].rolling(window=10).mean()
      df['30_week_SMA'] = df['Close'].rolling(window=30).mean()
      df['slope_raw'] = rolling_regression_slope(df['30_week_SMA'], window=10)
      df['slope_pct_per_week'] = df['slope_raw'] / df['30_week_SMA']          # fractional change per week
      df['SMA_Slope'] = df['slope_pct_per_week'] * 52 * 100       # % per year
      # Optional: also keep a simple angle if you still want it
      df['slope_angle_deg'] = np.degrees(np.arctan(df['slope_pct_per_week'] * 52))
      df['ATR'] = compute_atr(df, 10)
      df['OBV'] = compute_obv(df)
      df['OBV_Slope'] = df['OBV'].diff()
      df['30_week_avg_volume'] = df['Volume'].rolling(window=30).mean()
      # Calculate MACD and Signal Line
      df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
      df['21_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
      df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
      # Compute MACD Line
      df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
      # Compute Signal Line (9-day EMA of MACD Line)
      df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
      df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']


      # Compute raw EFI
      df['EFI'] = (df['Close'].diff()) * df['Volume']
      # Compute EMA of EFI
      df['EFI_EMA'] = df['EFI'].ewm(span=13, adjust=False).mean()
      # Determine if EFI_EMA is rising or falling
      df['EFI_EMA_Trend'] = df['EFI_EMA'].diff().apply(lambda x: 'Rising' if x > 0 else 'Falling')
      # Calculate ADX, +DMI and -DMI
      high = df['High']
      low = df['Low']
      close = df['Close']
      # Calculate directional movements
      up_move = high.diff()
      down_move = -low.diff()
      plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
      minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
      # Calculate True Range (TR)
      tr1 = high - low
      tr2 = (high - close.shift()).abs()
      tr3 = (low - close.shift()).abs()
      tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
      # Smooth TR, +DM, and -DM using Wilder’s smoothing
      atr = tr.rolling(window=14).sum()
      plus_dm_series = pd.Series(plus_dm.ravel(), index=df.index).astype(float)
      minus_dm_series = pd.Series(minus_dm.ravel(), index=df.index).astype(float)
      plus_dm_smoothed = plus_dm_series.rolling(window=14).sum()
      minus_dm_smoothed = minus_dm_series.rolling(window=14).sum()
      # Directional Indicators
      plus_di = 100 * (plus_dm_smoothed / atr)
      minus_di = 100 * (minus_dm_smoothed / atr)
      # DX and ADX
      dx = 100 * (np.abs(plus_di - minus_di) / (plus_di + minus_di))
      adx = dx.rolling(window=14).mean()
      # Add results to original DataFrame
      df['+DI'] = plus_di
      df['-DI'] = minus_di
      df['ADX'] = adx
      df['di_flag'] = df['-DI'] > df['+DI']
      df['di_flag'] = df['di_flag'].astype(int)
      df['adx_indicator'] = np.where(df['ADX'] > 15, 1, 0)
      df['adx_signal'] = df['adx_indicator'] * df['di_flag']

      # --- Overhead Resistance Filter ---
      recent_52_weeks = df[-52:]
      min_close_52w = recent_52_weeks['Close'].min()
      last_close = df['Close'].iloc[-1].iloc[0]
      # --- Above 52 weeks Low ---
      df['above_52w_low'] = last_close > min_close_52w
      df['below_52w_low'] = last_close < min_close_52w

      return df
  except Exception as e:
      print("There is an error getting weekly data", e)

def is_macd_bullish(df):
    """
    Determines if there is a bullish signal on the MACD indicator.
    A bullish signal occurs when the MACD line is above the signal line.

    Parameters:
    - df: dataframe. Must have columns 'MACD_Line' and 'Signal_Line'.
    Returns:
    - bool: True if a bullish signal is detected, otherwise False.
    """
    try:
      if df.empty:
        return False
      # Calculate MACD histogram if not already present

      curr = df.iloc[-3:]
      macd_crossover = curr['MACD_Line'].iloc[-1] < curr['Signal_Line'].iloc[-1]
      below_zero_line = curr['MACD_Line'].iloc[-1] < 0


      return macd_crossover, below_zero_line

    except Exception as e:
      print("Something went wrong while computing the MACD", e)

def calculate_vwap(df):
  """ Calculates vwap"""
  try:
    Typical_Price = (df['Close'].values + df['High'].values + df['Low'].values) / 3
    TPV = Typical_Price * df['Volume'].values
    vwap = TPV.cumsum() / df['Volume'].values.cumsum()
    return vwap
  except Exception as e:
    print("Something went wrong while computing the VWAP:", e)
    return None


def calculate_ma(data, length=10, ma_type="WMA"):
    if ma_type == "SMA":
        return data.rolling(window=length).mean()
    elif ma_type == "EMA":
        return data.ewm(span=length, adjust=False).mean()
    elif ma_type == "WMA":
        weights = np.arange(1, length+1)
        return data.rolling(length).apply(lambda x: np.dot(x, weights)/weights.sum(), raw=True)
    elif ma_type == "VWMA":
        return ta.volume_weighted_average_price(data, length)


# Function to compute ATR (Average True Range)
def compute_atr(df, period=10):
  try:
    df['High-Low'] = df['High'] - df['Low']
    df['High-Close'] = abs(df['High'] - df['Close'].shift(1))
    df['Low-Close'] = abs(df['Low'] - df['Close'].shift(1))
    df['TR'] = df[['High-Low', 'High-Close', 'Low-Close']].max(axis=1)
    return df['TR'].rolling(window=period).mean()
  except Exception as e:
      print("Something went wrong whilecomputing the ATR", e)


# Function to compute On-Balance Volume (OBV)
def compute_obv(df):
  try:
    # Calculate daily price change: 1 if price is up, -1 if down, 0 if unchanged
    price_change = df['Close'].diff()

    # Use price change to decide whether to add or subtract volume
    obv = (price_change > 0).astype(int) * df['Volume']  # Volume when price goes up
    obv -= (price_change < 0).astype(int) * df['Volume']  # Volume when price goes down

    # We accumulate the OBV by taking the cumulative sum of the volume changes
    obv = obv.cumsum()

    return obv
  except Exception as e:
      print("Something went wrong while computing the OBV", e)


# Function to calculate risk-reward ratio
def calculate_risk_reward(df):
  try:
    if df.empty or len(df) < 20:  # Ensure there are enough data points
        return np.nan

    latest_price = df['Close'].iloc[-1]

    # Use the ATR for setting support level
    atr = df['ATR'].iloc[-1]  # Latest ATR value
    price_ema = df['8_day_EMA'].iloc[-1]
    atr_multiple = 1.55 # You can adjust this multiplier based on your strategy

    # Calculate the support level using the ATR
    trailing = atr
    #stop      = price_ema + (trailing * atr_multiple)

    return trailing, price_ema
  except Exception as e:
      print("Something went wrong while computing the reward-risk ratio", e)

# Function to fetch daily data
def get_daily_data(ticker):
    df = yf.download(ticker, period="1y", interval="1d",auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level
    df['20_day_SMA'] = df['Close'].rolling(window=20).mean()
    df['50_day_avg_volume'] = df['Volume'].rolling(window=50).mean()
    df['8_day_EMA'] = df['Close'].ewm(span=8, adjust=False).mean()
    df['15_day_EMA'] = df['Close'].ewm(span=15, adjust=False).mean()
    df['21_day_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
    df['26_day_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
    df['5_day_SMA'] = df['Close'].rolling(window=5).mean()
    df['50_day_SMA'] = df['Close'].rolling(window=50).mean()
    df['trend_break'] = df['Close'] < df['Close'].rolling(5).min().shift(1)
    df['swing_high'] = df['High'].where(df['trend_break']).ffill()

    # Slope for 5-day SMA (short-term trend)
    df['slope5_raw'] = rolling_regression_slope(df['5_day_SMA'], window=10)
    df['slope5_pct_per_day'] = df['slope5_raw'] / df['5_day_SMA']  # fractional change per day
    df['slope5_annualized_pct'] = df['slope5_pct_per_day'] * 252 * 100  # % per year
    # Slope for 50-day SMA (intermediate-term trend)
    df['slope50_raw'] = rolling_regression_slope(df['50_day_SMA'], window=10)
    df['slope50_pct_per_day'] = df['slope50_raw'] / df['50_day_SMA']  # fractional change per day
    df['slope50_annualized_pct'] = df['slope50_pct_per_day'] * 252 * 100  # % per year
    df['100_day_SMA'] = df['Close'].rolling(window=100).mean()
    df['200_day_SMA'] = df['Close'].rolling(window=200).mean()
    df['ATR'] = compute_atr(df, 10)
    df["8EMA_minus_ATR"] = df["8_day_EMA"] - 0.7* df["ATR"]
    df["8EMA_minus_ATRL"] = df["8_day_EMA"] - 1.1* df["ATR"]
    # 1️⃣ Yesterday touched or pierced 8 EMA
    df['prior_touch_8ema'] = df['Low'].shift(1) <= df['8_day_EMA'].shift(1)
    # 2️⃣ Today closes above 8 EMA
    df['close_above_8ema'] = df['Close'] > df['8_day_EMA']
    # 3️⃣ Today closes above yesterday’s close (price rising)
    df['close_above_prev_close'] = df['Close'] > df['Close'].shift(1)
    # 4️⃣ Strong confirmation: Break previous high
    df['break_prev_high'] = df['Close'] > df['High'].shift(1)
    # 5️⃣ 8 EMA slope positive (trend filter)
    df['ema8_rising'] = df['8_day_EMA'] > df['8_day_EMA'].shift(1)
    # EMA 8 slope
    df['ema8_slope'] = df['8_day_EMA'] - df['8_day_EMA'].shift(1)
    # EMA 8 slope previous
    df['ema8_slope_prev'] = df['ema8_slope'].shift(1)
    # EMA 8 acceleration
    df['ema8_accel'] = df['ema8_slope'] - df['ema8_slope_prev']
    # EMA 8 slope direction (1 = up, 0 = down)
    df['ema8_dir'] = (df['ema8_slope'] > 0).astype(int)
    # Count direction changes over last 5 days
    df['ema8_direction_changes'] = (
      df['ema8_dir']
      .diff()
      .abs()
      .rolling(5)
      .sum()
      )
    # from here
    # 2. Avoid strong counter-trend moves and chop
    df['daily_return'] = df['Close'].pct_change()
    avoid_strong_up = df['daily_return'] > 0.035          # Strong bullish day
    avoid_chop = df['ema8_direction_changes'] >= 3
    # 3. EMA 8 Price Action
    df['touched_ema8']      = df['High'] >= df['8_day_EMA'] * 0.995
    df['closed_below_ema8'] = df['Close'] < df['8_day_EMA']
    df['bearish_candle']    = df['Close'] < df['Open']
    df['lower_low'] = df['Low'] < df['Low'].shift(1) * 0.995  # 0.5% lower

    # Strong upper wick rejection
    df['upper_wick_ratio']  = (df['High'] - df['Close']) / (df['High'] - df['Low'] + 0.0001)
    df['strong_upper_wick'] = df['upper_wick_ratio'] > 0.60

    # 4. Advanced Bearish Patterns
    df['bearish_engulfing'] = (
      (df['Close'] < df['Open']) &
      (df['Open'] > df['Close'].shift(1)) &
      (df['Close'] < df['Close'].shift(1))
    )

    # Intraday failed break above (single candle)
    df['failed_break_above_intraday'] = (
       (df['High'] > df['8_day_EMA']) &      # today's high pierced EMA
       (df['Close'] < df['8_day_EMA'])        # but closed back below
    )

    # Two-day version
    df['failed_break_above_twoday'] = (
       (df['Close'].shift(1) > df['8_day_EMA'].shift(1)) &
       (df['Close'] < df['8_day_EMA'])
    )

    df['failed_break_ema8'] = (
      df['failed_break_above_intraday'] |
      df['failed_break_above_twoday']
    )

    df['near_50sma'] = df['High'] >= df['50_day_SMA'] * 0.99
    df['weak_close'] = (df['Close'] - df['Low']) / (df['High'] - df['Low'] + 1e-6) < 0.4

    # 5. A and A+ Setups
    df['A_setup'] = (
      df['touched_ema8'] &
      df['closed_below_ema8'] &
      df['bearish_candle'] &
      df['strong_upper_wick']
      #df['weak_close']
    )

    df['A_plus_setup'] = (
      df['failed_break_ema8'] &
      #(df['bearish_engulfing'] | df['strong_upper_wick']) &
      df['bearish_engulfing'] &
      df['closed_below_ema8'] &
      (df['near_50sma'] | df['strong_upper_wick'])
    )

    # Trend Continuation / Momentum Trades ---
    # NOTE: Entry requires price to be within 1 ATR of 8 EMA (handled upstream)
    df['trend_continuation'] = (
      (df['Close'] < df['8_day_EMA']) &                     # Below EMA
      (df['Close'].shift(1) < df['8_day_EMA'].shift(1)) &   # Was already below
      #(df['High'].shift(1) >= df['8_day_EMA'].shift(1)) &  # Optional: rejection wick
      df['lower_low'] &                                     # Making lower lows
      (df['ema8_slope'] < 0)                                # EMA sloping down
      & (df['daily_return'] < -0.005)                       # strong bearish momentum
    )

    # 6. Entry Trigger (Momentum)
    df['entry_trigger'] = df['Low'] < df['Low'].shift(1) * 0.995  # 0.5% lower
    # --- Trend ---
    trend_short = df['slope50_raw'] < 0

    # ====================== FINAL SHORT SIGNAL ======================
    df['short_signal'] = (
        trend_short &
        (~avoid_strong_up) &
        (~avoid_chop) &
        (
          df['A_setup'] |
          df['A_plus_setup'] |
          (df['trend_continuation'] & df['entry_trigger'])  # entry trigger only for continuation
        )
    )


    # Signal Strength Labeling
    df['signal_type'] = 'None'
    df.loc[df['short_signal'] & (df['A_setup'] | df['A_plus_setup']) & df['trend_continuation'], 'signal_type'] = 'Hybrid'
    df.loc[df['short_signal'] & (df['A_setup'] | df['A_plus_setup']) & ~df['trend_continuation'], 'signal_type'] = 'Pullback (A/A+)'
    df.loc[df['short_signal'] & df['trend_continuation'] & ~(df['A_setup'] | df['A_plus_setup']), 'signal_type'] = 'Trend Continuation'

    df['signal_strength'] = 'None'
    # Pure A+
    df.loc[df['short_signal'] & df['A_plus_setup'] & ~df['trend_continuation'],'signal_strength' ] = 'A+'
    # Then A
    df.loc[df['short_signal'] & df['A_setup'] & ~df['A_plus_setup'], 'signal_strength'] = 'A'
    # Then B+ (only if NOT A or A+)
    df.loc[df['short_signal'] & df['trend_continuation'] & ~(df['A_setup'] | df['A_plus_setup']),'signal_strength'] = 'B+'
    # Hybrid — both pullback and continuation aligning
    df.loc[df['short_signal'] & (df['A_setup'] | df['A_plus_setup']) & df['trend_continuation'], 'signal_strength'] = 'A+'

    # Calculate MACD and Signal Line
    df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
    # Compute MACD Line
    df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
    # Compute Signal Line (9-day EMA of MACD Line)
    df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']
    # Compute raw EFI
    df['EFI'] = (df['Close'].diff()) * df['Volume']
    # Compute EMA of EFI
    df['EFI_EMA'] = df['EFI'].ewm(span=13, adjust=False).mean()
    # Determine if EFI_EMA is rising or falling
    df['EFI_EMA_Trend'] = df['EFI_EMA'].diff().apply(lambda x: 'Rising' if x > 0 else 'Falling')
    # Calculate ADX, +DMI and -DMI
    high = df['High']
    low = df['Low']
    close = df['Close']
    # Calculate directional movements
    up_move = high.diff()
    down_move = -low.diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    # Calculate True Range (TR)
    tr1 = high - low
    tr2 = (high - close.shift()).abs()
    tr3 = (low - close.shift()).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    # Smooth TR, +DM, and -DM using Wilder’s smoothing
    atr = tr.rolling(window=10).sum()
    plus_dm_series = pd.Series(plus_dm.ravel(), index=df.index).astype(float)
    minus_dm_series = pd.Series(minus_dm.ravel(), index=df.index).astype(float)
    plus_dm_smoothed = plus_dm_series.rolling(window=10).sum()
    minus_dm_smoothed = minus_dm_series.rolling(window=10).sum()
    # Directional Indicators
    plus_di = 100 * (plus_dm_smoothed / atr)
    minus_di = 100 * (minus_dm_smoothed / atr)
    # DX and ADX
    dx = 100 * (np.abs(plus_di - minus_di) / (plus_di + minus_di))
    adx = dx.rolling(window=10).mean()
    # Add results to original DataFrame
    df['+DI'] = plus_di
    df['-DI'] = minus_di
    df['ADX'] = adx
    df['di_flag'] = df['-DI'] > df['+DI']
    df['di_flag'] = df['di_flag'].astype(int)
    df['adx_indicator'] = np.where(df['ADX'] > 20, 1, 0)
    df['adx_signal'] = df['adx_indicator'] * df['di_flag']
    return df

# Function to fetch hourly data
def get_30mins_data(ticker):
    df = yf.download(ticker, interval='30m', period='60d',auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)  # keep only first level

    df['65d_SMA'] = df['Close'].rolling(window=65).mean()
    # Slope calculation: regression over recent 10 bars (~5 trading hours)
    df['slope_raw'] = rolling_regression_slope(df['65d_SMA'], window=10)
    # Normalize → fractional change per 30-minute bar
    df['slope_norm'] = df['slope_raw'] / df['65d_SMA']
    # Annualize to % per year (standard for 30m regular hours)
    bars_per_year = 252 * 13
    df['SMA_Slope'] = df['slope_norm'] * bars_per_year * 100
    # Short-term & medium-term moving averages
    df['21_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
    df['50_EMA'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['200_EMA'] = df['Close'].ewm(span=200, adjust=False).mean()
    # Calculate MACD and Signal Line
    df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
    # Compute MACD Line
    df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
    # Compute Signal Line (9-day EMA of MACD Line)
    df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']

    return df


# Function to fetch hourly data
def get_15min_data(ticker):
    df = yf.download(ticker, interval='15m', period='30d',auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)  # keep only first level
    df['130d_SMA'] = df['Close'].rolling(window=130).mean()
    # Slope calculation: regression over recent 10 bars (~5 trading hours)
    df['slope_raw'] = rolling_regression_slope(df['130d_SMA'], window=10)
    # Normalize → fractional change per 15-minute bar
    df['slope_norm'] = df['slope_raw'] / df['130d_SMA']
    # Annualize to % per year (standard for 30m regular hours)
    bars_per_year = 252 * 26
    df['SMA_Slope'] = df['slope_norm'] * bars_per_year * 100
    return df

def get_heikin_ashi_signal(ticker="AAPL", period="6mo", interval="1d"):
    # Fetch OHLC data
    df = yf.download(ticker, period=period, interval=interval, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level

    # Compute Heikin Ashi candles
    ha_df = pd.DataFrame(index=df.index)
    ha_df['HA_Close'] = (df['Open'] + df['High'] + df['Low'] + df['Close']) / 4

    ha_open = []
    for i in range(len(df)):
        if i == 0:
             ha_open.append((df['Open'].iloc[i] + df['Close'].iloc[i]) / 2)
        else:
            ha_open.append((ha_open[i-1] + ha_df['HA_Close'].iloc[i-1]) / 2)
    ha_df['HA_Open'] = ha_open
    #ha_df['HA_High'] = ha_df[['HA_Open', 'HA_Close']].assign(High=df['High']).max(axis=1)
    #ha_df['HA_Low'] = ha_df[['HA_Open', 'HA_Close']].assign(Low=df['Low']).min(axis=1)
    ha_df['HA_High'] = pd.concat([ha_df['HA_Open'], ha_df['HA_Close'], df['High']], axis=1).max(axis=1)
    ha_df['HA_Low'] = pd.concat([ha_df['HA_Open'], ha_df['HA_Close'], df['Low']], axis=1).min(axis=1)

    # Combine with original
    df = df.join(ha_df)
    # Check for green candle with flat bottom
    last = df.iloc[-1]
    red_candle = last['HA_Close'] < last['HA_Open']
    flat_top = abs(last['HA_High'] - last['HA_Open']) < 0.001 * last['HA_Open']
    signal = red_candle and flat_top

    print(f"\n🔍 Checking {ticker} ({interval} timeframe)")
    print(f"HA_Open: {last['HA_Open']:.2f}, HA_Close: {last['HA_Close']:.2f}, HA_Low: {last['HA_Low']:.2f}")
    if signal:
        print("✅ Heikin Ashi candle is Red with a FLAT top — strong bearish signal!")
    elif red_candle:
        print("🟢 Candle is red but not flat-topped — still bearish, but less strong.")
    else:
        print("🔴 Not a bearish candle — no entry confirmation yet.")

    return signal, red_candle

# Function to check monthly trend
def is_monthly_trend_bearish(df):
    if df.empty:
        return False

    adx_ok    = df['adx_signal'].iloc[-1] == 1
    latest_price = df['Close'].iloc[-1].iloc[0]
    latest_sma = df['10_month_SMA'].iloc[-1]
    #sma_slope  = df['SMA_Slope'].iloc[-1]< -0.1
    macd_bearish_signal, below_zero_line = is_macd_bullish(df)
    below_10_month_SMA = (latest_price < latest_sma)
    return below_10_month_SMA and adx_ok and macd_bearish_signal


# Function to check weekly trend
def is_weekly_trend_bearish(df2):
    if df2.empty:
        return False

    df = df2.copy()
    latest_price  = df['Close'].iloc[-1].iloc[0]
    latest_10sma  = df['10_week_SMA'].iloc[-1]
    below_10w_SMA = latest_price < latest_10sma
    latest_30sma  = df['30_week_SMA'].iloc[-1]
    below_30w_SMA = latest_price < latest_30sma
    sma_slope = df['SMA_Slope'].iloc[-1]< -10
    macd_bearish_signal,below_zero_line = is_macd_bullish(df)
    adx_ok        = df['adx_signal'].iloc[-1] == 1
    trend_ok      = below_10w_SMA and below_30w_SMA and adx_ok and sma_slope
    elderforce_trend_ok = df['EFI_EMA_Trend'].iloc[-1] == 'Falling'
    elderforce_ema_ok = df['EFI_EMA'].iloc[-1] < 0

    time.sleep(2)  # Add a delay of 1 second between requests

    return  trend_ok and macd_bearish_signal


# Function to check daily entry signal
def is_daily_entry_bearish(df2):
    if df2.empty:
        return False

    #df2 = generate_long_signal(df2)
    #df2 = generate_short_signal(df2)
    df = df2.copy()

    counter_trend_short_signal = df['short_signal'].iloc[-1]
    latest_price = df['Close'].iloc[-1].iloc[0]
    latest_50sma = df['50_day_SMA'].iloc[-1]
    latest_100sma = df['100_day_SMA'].iloc[-1]
    latest_200sma = df['200_day_SMA'].iloc[-1]
    below_50sma = latest_price < latest_50sma
    below_100sma = latest_price < latest_100sma
    below_200sma = latest_price < latest_200sma
    is_50sma_below_100sma = latest_50sma < latest_100sma
    is_50sma_below_200sma = latest_50sma < latest_200sma
    is_100sma_below_200sma = latest_100sma < latest_200sma
    sma_slope_5  = df['slope5_annualized_pct'].iloc[-1] < -25
    sma_slope_50 = df['slope50_annualized_pct'].iloc[-1] < -15
    elderforce_trend_ok = df['EFI_EMA_Trend'].iloc[-1] == 'Falling'
    elderforce_ema_ok = df['EFI_EMA'].iloc[-1] < 0
    macd_bearish_signal, below_zero_line = is_macd_bullish(df)
    adx_ok = df['adx_signal'].iloc[-1] == 1
    slopes_ok =  sma_slope_5 and sma_slope_50
    moving_averages_ok = below_50sma and below_100sma and is_50sma_below_100sma \
                         and below_200sma and is_100sma_below_200sma and is_50sma_below_200sma


    # Look for a breakout above 20-day SMA & RSI > 50
    return moving_averages_ok and adx_ok and sma_slope_50 and elderforce_ema_ok #and counter_trend_short_signal

# Check entry conditions
def check_entry_conditions(tickers):
    results = []
    for ticker in tickers:
      df = get_daily_data(ticker)
      latest_price        = df['Close'].iloc[-1]
      latest_sma          = df['50_day_SMA'].iloc[-1]
      latest_price_8ema   =  df['8_day_EMA'].iloc[-1]
      price_threshold_ATR = df['8EMA_minus_ATR'].iloc[-1]
      price_threshold_ATRL = df['8EMA_minus_ATRL'].iloc[-1]
      atr                  = df['ATR'].iloc[-1]
      signal_strength      = df['signal_strength'].iloc[-1]
      # Define tight A-Line band
      aline_lower          = latest_price_8ema - 0.3 * atr

      mfi_signal          = money_flow_signals(df)
      # Print results
      print(f"\nMoney Outflow indicator for {ticker} is:")
      print(not(mfi_signal))

      df_entry             = get_30mins_data(ticker)
      latest_priceh        = df_entry['Close'].iloc[-1]
      latest_priceh_5sma   = df_entry['65d_SMA'].iloc[-1]
      slope_hr             = df_entry['SMA_Slope'].iloc[-1] < -5
      priceh_buy           = latest_priceh < latest_priceh_5sma
      HA_sell_signal_h,rc_h= get_heikin_ashi_signal(ticker, period="90d", interval="1d")

      df_refined_entry     = get_15min_data(ticker)
      latest_pricem        = df_refined_entry['Close'].iloc[-1]
      latest_pricem_5sma   = df_refined_entry['130d_SMA'].iloc[-1]
      pricem_buy           = latest_pricem < latest_pricem_5sma
      slope_m              = df_refined_entry['SMA_Slope'].iloc[-1] < -5

      # Strict entry — flat top red HA required
      refined_entry_signal_strict = (
          slope_hr and slope_m and
          priceh_buy and pricem_buy and
          HA_sell_signal_h       )

      # Standard entry — just red HA candle required
      refined_entry_signal_standard = (
          slope_hr and slope_m and
          priceh_buy and pricem_buy and
          rc_h   )

      # Use strict for A+ signals, standard for A/B+
      if signal_strength == 'A+' :
        refined_entry_signal = refined_entry_signal_strict
      else:
        refined_entry_signal = refined_entry_signal_standard

      if latest_price <  price_threshold_ATRL:
        entry_signal = "Extended Short Entry"  ## > 1.1 ATR (too stretched)
      elif latest_price < price_threshold_ATR:
          if refined_entry_signal:
             entry_signal = "True Trend Short Entry"
          else:
             entry_signal = "Skip"
      elif aline_lower <= latest_price <= latest_price_8ema:
        entry_signal = "Aline Short Entry"
      else:
        entry_signal = "Skip"
      results.append([ticker, entry_signal])
    # Convert results to DataFrame
    df_results = pd.DataFrame(results, columns=["Asset", "Entry_Signal"])
    return df_results

# Multi-timeframe strategy check returning a DataFrame
def check_mtf_entry(tickers):
    results = []

    for ticker in tickers:
        monthly_df = get_monthly_data(ticker)
        weekly_df = get_weekly_data(ticker)
        daily_df = get_daily_data(ticker)

        if is_monthly_trend_bearish(monthly_df) and is_weekly_trend_bearish(weekly_df):
            if  True : #is_daily_entry_bearish(daily_df):
                entry_signal = "Bearish Entry Confirmed ✅"
            else:
                entry_signal = "No Bearish Entry Yet on Daily Timeframe ⏳"
        else:
            entry_signal = "Monthly/Weekly  Trend is not Bearish ❌"

        results.append([ticker, entry_signal])
        time.sleep(2)  # Add a delay of 1 second between requests

    # Convert results to DataFrame
    df_results = pd.DataFrame(results, columns=["Asset", "Entry_Signal"])
    return df_results



In [80]:
# Multi-time frame entry Check
etfs_to_check = etfs_clean

df_signals = check_mtf_entry(etfs_to_check)


df_final = df_signals[df_signals['Entry_Signal'] =="Bearish Entry Confirmed ✅"]

df_final.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,Asset,Entry_Signal
2,PRX.AS,Bearish Entry Confirmed ✅
6,SHL.AX,Bearish Entry Confirmed ✅
9,RMD.AX,Bearish Entry Confirmed ✅
10,LNW.AX,Bearish Entry Confirmed ✅
11,HVN.AX,Bearish Entry Confirmed ✅


## Generate Sell list

In [96]:
#df_final = df_signals[df_signals['Entry_Signal'] =="Bearish Entry Confirmed ✅"]
final_etfs_to_check = df_final['Asset'].tolist()
to_remove = ["PX"]
final_etfs_to_check = [x for x in final_etfs_to_check if x not in to_remove]
sell_list = check_entry_conditions(final_etfs_to_check)


sell_list= sell_list[sell_list['Entry_Signal'].isin(['Aline Short Entry', 'True Trend Short Entry'])]

sell_list

[*********************100%***********************]  1 of 1 completed



Money Outflow indicator for PRX.AS is:
True


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed



🔍 Checking PRX.AS (1d timeframe)
HA_Open: 41.69, HA_Close: 41.35, HA_Low: 40.76
🟢 Candle is red but not flat-topped — still bearish, but less strong.


[*********************100%***********************]  1 of 1 completed


...


Money Outflow indicator for CRM is:
True


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed



🔍 Checking CRM (1d timeframe)
HA_Open: 184.85, HA_Close: 180.06, HA_Low: 176.88
✅ Heikin Ashi candle is Red with a FLAT top — strong bearish signal!


[*********************100%***********************]  1 of 1 completed


,Asset,Entry_Signal
0,PRX.AS,Aline Short Entry
1,SHL.AX,True Trend Short Entry
4,HVN.AX,Aline Short Entry
5,FPH.AX,True Trend Short Entry
6,PODD,True Trend Short Entry
12,DHR,True Trend Short Entry
19,GIS,Aline Short Entry
20,CPB,Aline Short Entry
24,AON,Aline Short Entry
25,ICE,Aline Short Entry


# Find and filter correlated assets to reduce concentration risk.

In [97]:

def get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66, period="3mo", interval="1d"):
    """
    Filters a ranked list of tickers to return only uncorrelated picks.

    Parameters:
    -----------
    tickers : list
        All candidate tickers.
    ranked_picks : list
        Ranked list of tickers (best to worst).
    threshold : float
        Correlation threshold (default 0.66).
    period : str
        Data period for yfinance (default "3mo").
    interval : str
        Data interval (default "1d").

    Returns:
    --------
    final_selection : list
        List of uncorrelated tickers.
    corr_matrix : DataFrame
        Correlation matrix of daily returns.
    """
    # Step 1: Get prices
    data = yf.download(tickers, period=period, interval=interval,auto_adjust=True)["Close"]
    data = data.ffill()

    # Step 2: Convert to daily returns
    returns = data.pct_change().dropna()

    # Step 3: Correlation matrix
    corr_matrix = returns.corr()

    # Step 4: Filter uncorrelated picks
    final_selection = []
    for pick in ranked_picks:
        if all(abs(corr_matrix.loc[pick, sel]) <= threshold for sel in final_selection):
            final_selection.append(pick)

    return final_selection, corr_matrix



In [98]:
# Apply TA filters and prioritize ETFs
results = []
sell_list = sell_list[sell_list['Entry_Signal'].isin(['Aline Short Entry','True Trend Short Entry'])]


for etf in sell_list['Asset'].to_list():
   df           = get_daily_data(etf)
   price        = df['Close'].iloc[-1]
   swing_high   = df['swing_high'].iloc[-1]
   below_50sma  = price  < df['50_day_SMA'].iloc[-1]
   sma_slope_50 = df['slope50_annualized_pct'].iloc[-1]< 0
   vwap_df2     = anchored_vwap_old(etf, start_of_year)
   ytd_vwap     = vwap_df2['anchored_vwap'].iloc[-1]
   below_ytd_vwap = price < ytd_vwap
   print("Year to date VWAP is :", ytd_vwap)
   signal_strength = df['signal_strength'].iloc[-1]
   print("Signal Strength is :", signal_strength)
   signal_filter = (signal_strength == 'A+' or signal_strength == 'A' or signal_strength == 'B+')

   signal_type = df['signal_type'].iloc[-1]
   print("Signal Type is :", signal_type)

   #vwap_df     = anchored_vwap(etf, lookback_weeks=4)
   vwap_df     = anchored_vwap_structural(etf)
   vwap        = vwap_df['Anchored_VWAP'].iloc[-1]
   vwap_signal = vwap_df['Signal'].iloc[-1]
   below_vwap  = price < vwap
   print("Anchored VWAP from correction swing high is :", vwap)
   # MTD
   #vwap_mtd     = anchored_vwap_old(etf, first_day_month)
   #mtd_vwap    = vwap_mtd['anchored_vwap'].iloc[-1]
   #below_mtd_vwap = price < mtd_vwap
   #print("MTD VWAP is :", mtd_vwap)
   atr_multiple_map = {
     'A+': 1.55,
     'A':  1.55,
     'B+': 2.0,
     'C+': 2.0 }
   atr_multiple = atr_multiple_map.get(signal_strength, 1.55)
   # Only use swing high if it's within last 10 bars, otherwise fall back to ATR stop
   swing_high_age = df['trend_break'] .iloc[-10:].any()


   if sma_slope_50 and signal_filter and below_vwap:#and below_vwap :
    trail, price_ema = calculate_risk_reward(df)
    #trail = calculate_risk_reward(df)
    entry_price = price - max(0.25, 0.1*trail)
    stop_loss = price_ema + (atr_multiple*trail)
    if swing_high_age:
      stop = np.maximum(stop_loss, swing_high + 1*trail)
    else:
      stop = stop_loss # fall back to pure ATR stop
    risk = np.abs(stop - entry_price)
    take_profit = entry_price - (1.5*risk)
    reward =  entry_price - take_profit
    support_level = stop
    risk_reward_ratio = reward / risk
    # Ensure risk is greater than zero before division
    if risk > 0:

        rr_ratio  = reward / risk
    else:
        rr_ratio = np.nan

    stop_loss_perc = ((entry_price-support_level )/entry_price )*100
    take_profit_perc = (( entry_price - take_profit)/entry_price )*100
    # Fetch the Entry_Signal from buy_list
    entry_signal = sell_list.loc[sell_list['Asset'] == etf, 'Entry_Signal'].values[0]

    # Append results with Entry_Signal
    results.append({
            "Asset": etf,
            "Risk-Reward": rr_ratio,
            "Stop Loss": support_level,
            "Take Profit": take_profit,
            "Current Price": price,
            "Entry Price": entry_price,
            "Trail Price": trail,
            "Entry Signal": entry_signal,  # Add entry signal
            "stop_loss_perc": stop_loss_perc,
            "take_profit_perc": take_profit_perc,
            "Anchored VWAP": vwap,
            #"MTD VWAP": mtd_vwap
            "signal_type": signal_type,
            "signal_strength": signal_strength

        })

    time.sleep(2)  # Add a delay of 1 second between requests


# Sort ETFs by highest risk-to-reward ratio
try:
   df_results = pd.DataFrame(results).dropna().sort_values(by="Risk-Reward", ascending=True).reset_index(drop = True)
except Exception as e:
  print("No Asset to buy today, check back some other time!")
  df_results = pd.DataFrame({"Asset": ["No Asset available"]})

df2 = df_results.merge(df_o[['Asset','Type', 'score']], on='Asset', how='left')
df2['timestamp'] = datetime.now()
df2 = df2.sort_values(by='score', ascending=True)
df2.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Year to date VWAP is : 48.88020851660399
Signal Strength is : A
Signal Type is : Pullback (A/A+)
PRX.AS | AVWAP anchored from 2026-04-30 (structural LL @ 39.85)
Anchored VWAP from correction swing high is : 41.27576781606009


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Year to date VWAP is : 22.522608670318615
Signal Strength is : None
Signal Type is : None


[*********************100%***********************]  1 of 1 completed

...

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Year to date VWAP is : 136.22618216133088
Signal Strength is : None
Signal Type is : None
NOW | AVWAP anchored from 2026-05-06 (structural LL @ 88.21)
Anchored VWAP from correction swing high is : 91.47229588046034


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Year to date VWAP is : 226.1663554464472
Signal Strength is : None
Signal Type is : None
CRM | AVWAP anchored from 2026-04-30 (structural LL @ 173.00)
Anchored VWAP from correction swing high is : 182.40609918427955


,Asset,Risk-Reward,Stop Loss,Take Profit,Current Price,Entry Price,Trail Price,Entry Signal,stop_loss_perc,take_profit_perc,Anchored VWAP,signal_type,signal_strength,Type,score,timestamp
1,PODD,1.5,182.017796,109.639566,154.160004,153.066504,10.934998,True Trend Short Entry,-18.914192,28.371287,157.184697,Pullback (A/A+),A,Stock,-2.39,2026-05-10 01:15:47.928059
2,DHR,1.5,182.354311,153.290542,171.160004,170.728804,4.312000,True Trend Short Entry,-6.809342,10.214013,176.831609,Hybrid,A+,Stock,-0.95,2026-05-10 01:15:47.928059
3,GIS,1.5,36.495311,31.332034,34.680000,34.430000,0.997000,Aline Short Entry,-5.998579,8.997869,34.969343,Pullback (A/A+),A,Stock,-0.95,2026-05-10 01:15:47.928059
0,PRX.AS,1.5,43.096810,37.254781,41.009998,40.759998,1.142999,Aline Short Entry,-5.733100,8.599650,41.275768,Pullback (A/A+),A,AS,-0.56,2026-05-10 01:15:47.928059


## Sentiment Score

In [99]:
NEWS_API_KEY = "15c99612003d4971ad86698b50ed0bd7"  # Get one free from https://newsapi.org/
LOOKBACK_DAYS = 3

# Fetch recent news
def fetch_news(ticker, lookback_days=3):
    url = f"https://newsapi.org/v2/everything?q={ticker}&language=en&from={(datetime.now() - timedelta(days=lookback_days)).date()}&apiKey={NEWS_API_KEY}"
    resp = requests.get(url).json()
    if "articles" not in resp:
        return []
    return [a["title"] for a in resp["articles"]]

# Finbert Sentiment Scoring
finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert")

def get_sentiment_scores(news_list):
    if not news_list:
        return 0
    results = finbert(news_list)
    time.sleep(2)  # Add a delay of 1 second between requests
    scores = [1 if r["label"] == "positive" else -1 if r["label"] == "negative" else 0 for r in results]
    return np.mean(scores)

def build_sentiment_table(TICKERS):
    records = []
    for ticker in TICKERS:
        print(f"Processing {ticker}...")
        news = fetch_news(ticker, LOOKBACK_DAYS)
        sentiment_score = get_sentiment_scores(news)
        combined = {
            "Ticker": ticker,
            "Sentiment": sentiment_score
        }
        records.append(combined)
    df = pd.DataFrame(records)

    # Weighted score (adjustable)
    df["Composite_Score"] = (

        df["Sentiment"].rank(pct=True)
    )

    df = df.sort_values("Composite_Score", ascending=False).reset_index(drop=True)
    return df

# Run sentiment scoring
tickers = df2['Asset'].tolist()
results = build_sentiment_table(tickers)
top_assets = results[results["Sentiment"] <= 0]

top_assets.head()
#top_assets = tickers

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing PODD...
Processing DHR...
Processing GIS...
Processing PRX.AS...


,Ticker,Sentiment,Composite_Score
0,PODD,0.0,0.625
1,DHR,0.0,0.625
2,GIS,0.0,0.625
3,PRX.AS,0.0,0.625


# US Stock Entries (A-Line)

In [100]:
# SP 500 stocks
# Fetch the Entry_Signal from buy_list
try:
  df3 = df2[df2['Asset'].isin(top_assets['Ticker'])]
  sp500_stocks_dt = df3[(df3['Type'] == 'Stock') & (df3['Entry Signal'] == 'Aline Short Entry')].reset_index(drop=True)
  # Example usage
  tickers = sp500_stocks_dt['Asset'].tolist()  # Replace with your list of tickers
  ranked_picks = tickers

  final_extended_selection, corr_matrix = get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66)

  #print("Final uncorrelated picks:", final_extended_selection)
  print("\nCorrelation matrix:\n", corr_matrix)
  # Keep only rows where Asset is in filtered
  filtered_sp500_list = sp500_stocks_dt[sp500_stocks_dt["Asset"].isin(final_extended_selection)]

except Exception as e:
  print("No Asset to buy today, check back some other time!")
  filtered_sp500_list = pd.DataFrame({"Asset": ["No Asset available"]})



filtered_sp500_list

[*********************100%***********************]  1 of 1 completed


Correlation matrix:
 Ticker  GIS
Ticker     
GIS     1.0


,Asset,Risk-Reward,Stop Loss,Take Profit,Current Price,Entry Price,Trail Price,Entry Signal,stop_loss_perc,take_profit_perc,Anchored VWAP,signal_type,signal_strength,Type,score,timestamp
0,GIS,1.5,36.495311,31.332034,34.68,34.43,0.997,Aline Short Entry,-5.998579,8.997869,34.969343,Pullback (A/A+),A,Stock,-0.95,2026-05-10 01:15:47.928059


## US Stock Entries (True Trend Entry)

In [102]:
# SP 500 stocks
# Fetch the Entry_Signal from buy_list
try:
  #df3 = df2[df2['Asset'].isin(top_assets['Ticker'])]
  sp500_stocks = df3[(df3['Type'] == 'Stock') & (df3['Entry Signal'].isin(['True Trend Short Entry']))].reset_index(drop=True)

  # Example usage
  tickers = sp500_stocks['Asset'].tolist()  # Replace with your list of tickers
  ranked_picks = tickers

  final_aline_selection, corr_matrix = get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66)

  #print("Final uncorrelated picks:", final_aline_selection)
  print("\nCorrelation matrix:\n", corr_matrix)
  # Keep only rows where Asset is in filtered
  filtered_sp500AL_list = sp500_stocks[sp500_stocks["Asset"].isin(final_aline_selection)]


except Exception as e:
  print("No Asset to buy today, check back some other time!")
  filtered_sp500AL_list = pd.DataFrame({"Asset": ["No Asset available"]})

filtered_sp500AL_list

[*********************100%***********************]  2 of 2 completed


Correlation matrix:
 Ticker       DHR      PODD
Ticker                    
DHR     1.000000  0.215387
PODD    0.215387  1.000000


,Asset,Risk-Reward,Stop Loss,Take Profit,Current Price,Entry Price,Trail Price,Entry Signal,stop_loss_perc,take_profit_perc,Anchored VWAP,signal_type,signal_strength,Type,score,timestamp
0,PODD,1.5,182.017796,109.639566,154.160004,153.066504,10.934998,True Trend Short Entry,-18.914192,28.371287,157.184697,Pullback (A/A+),A,Stock,-2.39,2026-05-10 01:15:47.928059
1,DHR,1.5,182.354311,153.290542,171.160004,170.728804,4.312000,True Trend Short Entry,-6.809342,10.214013,176.831609,Hybrid,A+,Stock,-0.95,2026-05-10 01:15:47.928059


## Dutch Lag Cap Stock Entries (Aline Short Entry)

In [103]:
# AEX Small Capstocks
# Fetch the Entry_Signal from buy_list
try:
  aex_stocks = df3[(df3['Type'] == 'AS') & (df3['Entry Signal'].isin(['Aline Short Entry','True Trend Short Entry']))].reset_index(drop=True)
  # Example usage
  tickers = aex_stocks['Asset'].tolist()  # Replace with your list of tickers
  ranked_picks = tickers

  final_aline_selection, corr_matrix = get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66)

  #print("Final uncorrelated picks:", final_aline_selection)
  print("\nCorrelation matrix:\n", corr_matrix)
  # Keep only rows where Asset is in filtered
  filtered_aex_list = aex_stocks[aex_stocks["Asset"].isin(final_aline_selection)]

except Exception as e:
  print("No Asset to buy today, check back some other time!")
  filtered_aex_list = pd.DataFrame({"Asset": ["No Asset available"]})

filtered_aex_list

[*********************100%***********************]  1 of 1 completed


Correlation matrix:
 Ticker  PRX.AS
Ticker        
PRX.AS     1.0


,Asset,Risk-Reward,Stop Loss,Take Profit,Current Price,Entry Price,Trail Price,Entry Signal,stop_loss_perc,take_profit_perc,Anchored VWAP,signal_type,signal_strength,Type,score,timestamp
0,PRX.AS,1.5,43.09681,37.254781,41.009998,40.759998,1.142999,Aline Short Entry,-5.7331,8.59965,41.275768,Pullback (A/A+),A,AS,-0.56,2026-05-10 01:15:47.928059


## ASX Stock Entries (Aline Short Entry)

In [104]:
# ASX Small Capstocks
# Fetch the Entry_Signal from buy_list
try:
  asx_stocks = df3[(df3['Type'] == 'ASX') & (df3['Entry Signal'].isin(['Aline Short Entry','True Trend Short Entry']))].reset_index(drop=True)
  # Example usage
  tickers = asx_stocks['Asset'].tolist()  # Replace with your list of tickers
  ranked_picks = tickers
  final_aline_selection, corr_matrix = get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66)

  #print("Final uncorrelated picks:", final_aline_selection)
  print("\nCorrelation matrix:\n", corr_matrix)
  # Keep only rows where Asset is in filtered
  filtered_ASXAL_list = asx_stocks[asx_stocks["Asset"].isin(final_aline_selection)]
except Exception as e:
  print("No Asset to buy today, check back some other time!")
  filtered_ASXAL_list = pd.DataFrame({"Asset": ["No Asset available"]})

filtered_ASXAL_list

[*********************100%***********************]  0 of 0 completed

No Asset to buy today, check back some other time!


,Asset
0,No Asset available


## TSX Stock Entries (Aline Short Entry)

In [105]:
# ASX Small Capstocks
# Fetch the Entry_Signal from buy_list
try:
  tsx_stocks = df3[(df3['Type'] == 'TSX') & (df3['Entry Signal'].isin(['Aline Short Entry','True Trend Short Entry']))].reset_index(drop=True)
  # Example usage
  tickers = tsx_stocks['Asset'].tolist()  # Replace with your list of tickers
  ranked_picks = tickers
  final_aline_selection, corr_matrix = get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66)

  #print("Final uncorrelated picks:", final_aline_selection)
  print("\nCorrelation matrix:\n", corr_matrix)
  # Keep only rows where Asset is in filtered
  filtered_TSXAL_list = tsx_stocks[tsx_stocks["Asset"].isin(final_aline_selection)]
except Exception as e:
  print("No Asset to buy today, check back some other time!")
  filtered_TSXAL_list  = pd.DataFrame({"Asset": ["No Asset available"]})

filtered_TSXAL_list

[*********************100%***********************]  0 of 0 completed

No Asset to buy today, check back some other time!


,Asset
0,No Asset available
